In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pickle

from openood.networks import ResNet18_32x32, ResNet18_64x64
from openood.evaluation_api import Evaluator

In [ ]:
class TCLNet(nn.Module):
    """TCL 모델을 OpenOOD 평가 인터페이스에 맞게 래핑.

    이 체크포인트 (13_07_59) 구조:
      encoder_q : ResNet18_32x32 기반 _ResNet18_32x32_Encoder
                  (conv1→bn1→layer1-4→avgpool, fc/head는 inherited but unused in forward)
      classifier_q : Linear(512,512)→BN→ReLU→Dropout→Linear(512,10)→BN
    """

    def __init__(self, num_classes=10, dropout=0.25):
        super().__init__()
        self.encoder = ResNet18_32x32(num_classes=num_classes)
        self.classifier = nn.Sequential(
            nn.Linear(512, 512),         # 0
            nn.BatchNorm1d(512),         # 1
            nn.ReLU(inplace=True),       # 2 (no state)
            nn.Dropout(p=dropout),       # 3 (no state)
            nn.Linear(512, num_classes), # 4
            nn.BatchNorm1d(num_classes), # 5
        )
        self.feature_size = 512
        # rankfeat 등 postprocessor가 직접 접근하는 속성 노출
        self.fc = self.classifier[4]
        self.avgpool = self.encoder.avgpool
        self.layer4 = self.encoder.layer4

    def _encode(self, x):
        enc = self.encoder
        out = F.relu(enc.bn1(enc.conv1(x)))
        out = enc.layer1(out)
        out = enc.layer2(out)
        out = enc.layer3(out)
        out = enc.layer4(out)
        out = enc.avgpool(out)
        return out.view(out.size(0), -1)

    def forward(self, x, return_feature=False, return_feature_list=False):
        feature = self._encode(x)
        logits = self.classifier(feature)
        if return_feature:
            return logits, feature
        elif return_feature_list:
            enc = self.encoder
            f1 = F.relu(enc.bn1(enc.conv1(x)))
            f2 = enc.layer1(f1)
            f3 = enc.layer2(f2)
            f4 = enc.layer3(f3)
            f5 = enc.layer4(f4)
            return logits, [f1, f2, f3, f4, f5]
        return logits

    def forward_threshold(self, x, threshold):
        return self.classifier(self._encode(x).clip(max=threshold))

    def intermediate_forward(self, x, layer_index):
        enc = self.encoder
        out = F.relu(enc.bn1(enc.conv1(x)))
        out = enc.layer1(out)
        if layer_index == 1:
            return out
        out = enc.layer2(out)
        if layer_index == 2:
            return out
        out = enc.layer3(out)
        if layer_index == 3:
            return out
        out = enc.layer4(out)
        if layer_index == 4:
            return out
        raise ValueError

    def get_fc(self):
        fc = self.classifier[4]
        return fc.weight.cpu().detach().numpy(), fc.bias.cpu().detach().numpy()

    def get_fc_layer(self):
        return self.classifier[4]

In [ ]:
class TCLNet64x64(nn.Module):
    """TCL 모델 (ResNet18_64x64 기반)을 OpenOOD 평가 인터페이스에 맞게 래핑.

    TinyImageNet 체크포인트 구조:
      encoder_q : ResNet18_64x64 기반 _ResNet18_64x64_Encoder
                  (conv1→bn1→layer1-4→avgpool(8), fc/head는 inherited but unused in forward)
      classifier_q : Linear(512,512)→BN→ReLU→Dropout→Linear(512,num_classes)→BN
    """

    def __init__(self, num_classes=200, dropout=0.25):
        super().__init__()
        self.encoder = ResNet18_64x64(num_classes=num_classes)
        self.classifier = nn.Sequential(
            nn.Linear(512, 512),         # 0
            nn.BatchNorm1d(512),         # 1
            nn.ReLU(inplace=True),       # 2 (no state)
            nn.Dropout(p=dropout),       # 3 (no state)
            nn.Linear(512, num_classes), # 4
            nn.BatchNorm1d(num_classes), # 5
        )
        self.feature_size = 512
        self.fc = self.classifier[4]
        self.avgpool = self.encoder.avgpool
        self.layer4 = self.encoder.layer4

    def _encode(self, x):
        enc = self.encoder
        out = F.relu(enc.bn1(enc.conv1(x)))
        out = enc.layer1(out)
        out = enc.layer2(out)
        out = enc.layer3(out)
        out = enc.layer4(out)
        out = enc.avgpool(out)
        return out.view(out.size(0), -1)

    def forward(self, x, return_feature=False, return_feature_list=False):
        feature = self._encode(x)
        logits = self.classifier(feature)
        if return_feature:
            return logits, feature
        elif return_feature_list:
            enc = self.encoder
            f1 = F.relu(enc.bn1(enc.conv1(x)))
            f2 = enc.layer1(f1)
            f3 = enc.layer2(f2)
            f4 = enc.layer3(f3)
            f5 = enc.layer4(f4)
            return logits, [f1, f2, f3, f4, f5]
        return logits

    def forward_threshold(self, x, threshold):
        return self.classifier(self._encode(x).clip(max=threshold))

    def intermediate_forward(self, x, layer_index):
        enc = self.encoder
        out = F.relu(enc.bn1(enc.conv1(x)))
        out = enc.layer1(out)
        if layer_index == 1:
            return out
        out = enc.layer2(out)
        if layer_index == 2:
            return out
        out = enc.layer3(out)
        if layer_index == 3:
            return out
        out = enc.layer4(out)
        if layer_index == 4:
            return out
        raise ValueError

    def get_fc(self):
        fc = self.classifier[4]
        return fc.weight.cpu().detach().numpy(), fc.bias.cpu().detach().numpy()

    def get_fc_layer(self):
        return self.classifier[4]

In [ ]:
ckpt_path = '..'

net = TCLNet(num_classes=10)
raw_ckpt = torch.load(ckpt_path, map_location='cpu')

# encoder_q 로드 (fc/pseudo_linear은 forward에서 안 쓰이고 shape 불일치라 제외)
_skip = {'fc.weight', 'fc.bias', 'pseudo_linear.weight', 'pseudo_linear.bias'}
encoder_state = {k[len('module.encoder_q.'):]: v
                 for k, v in raw_ckpt.items()
                 if k.startswith('module.encoder_q.')
                 and k[len('module.encoder_q.'):] not in _skip}
net.encoder.load_state_dict(encoder_state, strict=False)

# classifier_q 로드 (Sequential index가 체크포인트와 동일: 0,1,4,5)
cls_state = {k[len('module.classifier_q.'):]: v
             for k, v in raw_ckpt.items()
             if k.startswith('module.classifier_q.')}
net.classifier.load_state_dict(cls_state, strict=False)

print('Encoder load OK')
print('Classifier load OK')

net.cuda()
net.eval()

# sanity check
with torch.no_grad():
    dummy = torch.randn(4, 3, 32, 32).cuda()
    out = net(dummy)
    print('Forward pass OK, output shape:', out.shape)

In [ ]:
OOD_list = ['msp', 'odin','ebo','gradnorm','react','mls','klm','vim','knn','dice','rankfeat','ash','she','mds','rmds','gram','mds_ensemble','temp_scaling','openmax']

res = []
accuracy = 0
for method in OOD_list:
    print(f'-------------------{method}----------------------')
    try:
        evaluator = Evaluator(
            net,
            id_name='cifar10',
            data_root='./data',
            config_root=None,
            preprocessor=None,
            postprocessor_name=method,
            postprocessor=None,
            batch_size=200,
            shuffle=False,
            num_workers=2)

        metrics = evaluator.eval_ood(fsood=False)
        near_auroc, far_auroc = metrics['AUROC']['nearood'], metrics['AUROC']['farood']
        near_aupr_in, far_aupr_in = metrics['AUPR_IN']['nearood'], metrics['AUPR_IN']['farood']
        near_aupr_out, far_aupr_out = metrics['AUPR_OUT']['nearood'], metrics['AUPR_OUT']['farood']
        accuracy = metrics['ACC']['nearood']
        res.append([method, near_auroc, far_auroc, near_aupr_in, far_aupr_in, near_aupr_out, far_aupr_out])

    except Exception as e:
        print(f'Error occurred with method {method}: {e}')
        res.append([method, 0, 0, 0, 0, 0, 0])

res.append(accuracy)

out_path = '.pkl'
with open(out_path, 'wb') as f:
    pickle.dump(res, f)

print(f'Results have been saved to {out_path}')